<a href="https://colab.research.google.com/github/uniesecruz/ESALQ_money_laundering/blob/HI-medium/TCC_ESALQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Instalação das bibliotecas

In [ ]:
# Instalar pyspark
!pip install pyspark

# Leitura do arquivos CSV com spark

In [ ]:
from pyspark.sql import SparkSession
import os

# 1. Definir a memória máxima para o driver (essencial no Colab)
# Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
memory_limit = "46g"

spark = SparkSession.builder \
    .appName("Max_Performance_Spark") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

SparkSession inicializada com foco em High-RAM (46g).


In [ ]:
# Carregar o arquivo CSV usando Spark
spark_trans_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Medium_Trans.csv', header=True, inferSchema=True)

print("Schema do Spark DataFrame de Transações:")
spark_trans_df.printSchema()

print("Primeiras 5 linhas do Spark DataFrame de Transações:")
spark_trans_df.show(5)

Schema do Spark DataFrame de Transações:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Transações:
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+--------------

In [ ]:
# Carregar o arquivo CSV de contas usando Spark
spark_accounts_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Medium_accounts.csv', header=True, inferSchema=True)

print("Schema do Spark DataFrame de Contas:")
spark_accounts_df.printSchema()

print("Primeiras 5 linhas do Spark DataFrame de Contas:")
spark_accounts_df.show(5)

Schema do Spark DataFrame de Contas:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Contas:
+--------------------+-------+--------------+-----------+--------------------+
|           Bank Name|Bank ID|Account Number|  Entity ID|         Entity Name|
+--------------------+-------+--------------+-----------+--------------------+
|     China Bank #561|  53267|     817D00980|2AA1F24F180| Corporation #183669|
|   Spain Bank #18657| 316997|     808BB2280|2AA1EEB8540| Partnership #193780|
|First Bank of Helena| 339367|     8505ED380|2AA206D7790|Sole Proprietorsh...|
|   Mexico Bank #3367|3148419|     8363D4180|2AA2001B1A0| Partnership #133577|
|Switzerland Bank ...|3174937|     842090C80|2AA20224CB0|Sole Proprietorsh...|
+--------------------+-------+--------------+-----------+--------

Salvando os arquivos em parquet

In [ ]:
spark_trans_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_Trans', mode='overwrite')
print("spark_trans_df salvo em parquet.")

spark_accounts_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_accounts', mode='overwrite')
print("spark_accounts_df salvo em parquet.")

spark_trans_df salvo em parquet.
spark_accounts_df salvo em parquet.


# Lendo o arquivo parquet (Começar aqui)

In [ ]:
# from pyspark.sql import SparkSession
# import os

# # 1. Definir a memória máxima para o driver (essencial no Colab)
# # Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
# memory_limit = "46g"

# spark = SparkSession.builder \
#     .appName("Max_Performance_Spark") \
#     .config("spark.driver.memory", memory_limit) \
#     .config("spark.executor.memory", memory_limit) \
#     .config("spark.driver.maxResultSize", "10g") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.3") \
#     .config("spark.ui.port", "4050") \
#     .getOrCreate()

# print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

SparkSession inicializada com foco em High-RAM (46g).


In [7]:
spark_trans_df= spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_Trans')
spark_accounts_df = spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_accounts')

# Profiling Trans

In [8]:
print("### Profiling do spark_trans_df ###")

print("\nSchema do DataFrame:")
spark_trans_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_trans_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_trans_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_trans_df.summary().show()

print(f"\nNúmero total de linhas: {spark_trans_df.count()}")

print("\nNomes das Colunas:")
print(spark_trans_df.columns)

### Profiling do spark_trans_df ###

Schema do DataFrame:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)


Tipos de Dados das Colunas:
Timestamp: string
From Bank: int
Account2: string
To Bank: int
Account4: string
Amount Received: double
Receiving Currency: string
Amount Paid: double
Payment Currency: string
Payment Format: string
Is Laundering: int

Estatísticas Descritivas (describe()):
+-------+----------------+-----------------+---------+------------------+---------+--------------------+------------------+--------------------+-----------------+----

# Profiling Accounts

In [9]:
print("### Profiling do spark_accounts_df ###")

print("\nSchema do DataFrame:")
spark_accounts_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_accounts_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_accounts_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_accounts_df.summary().show()

print(f"\nNúmero total de linhas: {spark_accounts_df.count()}")

print("\nNomes das Colunas:")
print(spark_accounts_df.columns)

### Profiling do spark_accounts_df ###

Schema do DataFrame:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)


Tipos de Dados das Colunas:
Bank Name: string
Bank ID: int
Account Number: string
Entity ID: string
Entity Name: string

Estatísticas Descritivas (describe()):
+-------+------------------+-----------------+--------------+-----------+--------------------+
|summary|         Bank Name|          Bank ID|Account Number|  Entity ID|         Entity Name|
+-------+------------------+-----------------+--------------+-----------+--------------------+
|  count|           2087786|          2087786|       2087786|    2087786|             2087786|
|   mean|              NULL|632102.3694339362|      Infinity|       NULL|                NULL|
| stddev|              NULL|970374.5529627781|           NaN|       NULL|             

# Join dos datasets

In [ ]:
from pyspark.sql import functions as F


# 2. Renomear colunas de trans_df (Equivalente ao trans_df.columns no Pandas)
# Nota: Como o Spark não aceita atribuição direta de lista em .columns, usamos select e alias
new_columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account',
               'Amount Received', 'Receiving Currency', 'Amount Paid',
               'Payment Currency', 'Payment Format', 'Is Laundering']

trans_df = trans_df.toDF(*new_columns)

# 3. Preparar accounts_df para os joins (selecionando apenas colunas necessárias)
# Isso evita colisões de nomes e facilita o mapeamento posterior
acc_info = accounts_df.select(
    F.col("Bank ID"),
    F.col("Account Number"),
    F.col("Bank Name"),
    F.col("Entity ID"),
    F.col("Entity Name")
)

# --- JOIN 1: Informações da conta de ORIGEM (Sender) ---
trans_enriched_df = trans_df.join(
    F.broadcast(acc_info),
    (trans_df["From Bank"] == acc_info["Bank ID"]) &
    (trans_df["From Account"] == acc_info["Account Number"]),
    how='left'
).select(
    trans_df["*"], # Mantém todas as colunas originais da transação
    F.col("Bank Name").alias("From Bank Name"),
    F.col("Entity ID").alias("From Entity ID"),
    F.col("Entity Name").alias("From Entity Name")
)

# --- JOIN 2: Informações da conta de DESTINO (Receiver) ---
trans_enriched_df = trans_enriched_df.join(
    F.broadcast(acc_info),
    (trans_enriched_df["To Bank"] == acc_info["Bank ID"]) &
    (trans_enriched_df["To Account"] == acc_info["Account Number"]),
    how='left'
).select(
    trans_enriched_df["*"], # Mantém as colunas do primeiro join
    F.col("Bank Name").alias("To Bank Name"),
    F.col("Entity ID").alias("To Entity ID"),
    F.col("Entity Name").alias("To Entity Name")
)

# Exibir o resultado
print("Tabela de Transações Enriquecida:")
trans_enriched_df.show(5)

# Se precisar contar o total (equivalente ao print final)
# print(f"Total de registros: {trans_enriched_df.count()}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Configuração de Schema Manual (Recomendado pelo artigo para evitar overhead)
trans_schema = StructType([
    StructField("Timestamp", StringType(), True),
    StructField("From Bank", StringType(), True),
    StructField("from_account", StringType(), True), # Renomeado na leitura
    StructField("To Bank", StringType(), True),
    StructField("to_account", StringType(), True),   # Renomeado na leitura
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True)
])

# 2. Carregamento dos Dados
# Dica: No HI-Medium, use 'from_account' e 'to_account' para evitar nomes duplicados
# spark_trans_df = spark.read.csv("HI-Medium_Trans.csv", header=True, schema=trans_schema)
# spark_accounts_df = spark.read.csv("HI-Medium_accounts.csv", header=True, inferSchema=True)

# 3. Double Join com Broadcasting (Técnica de Big Data para o TCC)
# O artigo destaca que o contexto da entidade é vital.
# Vamos trazer os dados da conta de ORIGEM e DESTINO.

# Preparando tabela de contas para o Join
accounts_info = spark_accounts_df.select(
    F.col("Account Number").alias("acc_id"),
    F.col("Entity Name").alias("entity"),
    F.col("Bank Name").alias("bank_name")
)

# Join para o Remetente (Sender)
df_enriched = spark_trans_df.join(
    F.broadcast(accounts_info), # Otimização crucial para HI-Medium
    spark_trans_df.from_account == accounts_info.acc_id,
    "left"
).select(spark_trans_df["*"],
         F.col("entity").alias("sender_entity"),
         F.col("bank_name").alias("sender_bank_name")
).drop("acc_id")

# Join para o Destinatário (Receiver)
df_final = df_enriched.join(
    F.broadcast(accounts_info),
    df_enriched.to_account == accounts_info.acc_id,
    "left"
).select(df_enriched["*"],
         F.col("entity").alias("receiver_entity"),
         F.col("bank_name").alias("receiver_bank_name")
).drop("acc_id")

# 4. Verificação de Integridade
print(f"Total de Transações: {df_final.count()}")
df_final.show(5)

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `from_account` is not supported.